# 現代數據處理工具 (2024-2025)

本 Notebook 介紹三個現代高效能數據處理工具，它們在處理大規模數據時比 Pandas 更快更高效：

1. **Polars** - 高性能 DataFrame 庫（Rust 實現）
2. **DuckDB** - 嵌入式分析型數據庫（OLAP）
3. **Vaex** - 核心外（Out-of-Core）DataFrame 庫

## 為什麼需要這些工具？

### Pandas 的限制
- ⚠️ 記憶體效率低（數據需完全載入記憶體）
- ⚠️ 單線程執行（無法充分利用多核 CPU）
- ⚠️ 處理大型數據集速度慢（>1GB）

### 現代工具的優勢
- ✅ 多線程並行處理
- ✅ 更高的記憶體效率
- ✅ 懶惰執行（Lazy Evaluation）
- ✅ 更快的查詢速度（10-100x）
- ✅ 原生支援更多數據格式

In [ ]:
# 安裝必要的套件
!pip install polars duckdb pyarrow -q
# Vaex 可選安裝: !pip install vaex -q

import polars as pl
import duckdb
import pandas as pd
import numpy as np
import time
from pathlib import Path

print(f"Polars version: {pl.__version__}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Polars - 高性能 DataFrame 庫

### 主要特點
- 🚀 使用 Rust 編寫，速度極快
- 🧵 自動多線程並行處理
- 💾 記憶體效率高（使用 Apache Arrow）
- 🔄 支援懶惰執行（Lazy Evaluation）
- 📝 語法清晰、類似 Pandas

### 何時使用 Polars？
- 處理 1GB - 100GB 的數據
- 需要快速的數據轉換和聚合
- 希望充分利用多核 CPU
- 從 Pandas 遷移（語法相似）

In [ ]:
# 創建示例數據
np.random.seed(42)
n_rows = 1_000_000

# 使用 Pandas 創建數據
df_pandas = pd.DataFrame({
    'id': range(n_rows),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n_rows),
    'value1': np.random.randn(n_rows),
    'value2': np.random.randn(n_rows) * 100,
    'timestamp': pd.date_range('2023-01-01', periods=n_rows, freq='1min')
})

# 轉換為 Polars DataFrame
df_polars = pl.DataFrame(df_pandas)

print(f"數據集大小: {n_rows:,} 行")
print(f"\nPandas DataFrame 記憶體使用: {df_pandas.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Polars DataFrame 記憶體使用: {df_polars.estimated_size('mb'):.2f} MB")

In [ ]:
# Polars 基本操作示例
print("=== Polars DataFrame 基本資訊 ===")
print(df_polars.head())
print("\n數據類型:")
print(df_polars.dtypes)
print("\n統計摘要:")
print(df_polars.describe())

In [ ]:
# 性能比較：分組聚合操作
print("=== 性能比較：分組聚合 ===")

# Pandas
start = time.time()
result_pandas = df_pandas.groupby('category').agg({
    'value1': ['mean', 'std', 'min', 'max'],
    'value2': ['sum', 'mean']
})
time_pandas = time.time() - start
print(f"Pandas 耗時: {time_pandas:.4f} 秒")

# Polars (Eager)
start = time.time()
result_polars = df_polars.group_by('category').agg([
    pl.col('value1').mean().alias('value1_mean'),
    pl.col('value1').std().alias('value1_std'),
    pl.col('value1').min().alias('value1_min'),
    pl.col('value1').max().alias('value1_max'),
    pl.col('value2').sum().alias('value2_sum'),
    pl.col('value2').mean().alias('value2_mean')
])
time_polars = time.time() - start
print(f"Polars 耗時: {time_polars:.4f} 秒")

print(f"\n⚡ Polars 比 Pandas 快 {time_pandas/time_polars:.2f}x")
print("\nPolars 結果:")
print(result_polars)

In [ ]:
# Polars 懶惰執行（Lazy Evaluation）
print("=== Polars 懶惰執行示例 ===")

# 構建查詢計劃（不立即執行）
lazy_query = (
    df_polars.lazy()
    .filter(pl.col('value1') > 0)
    .with_columns([
        (pl.col('value1') * pl.col('value2')).alias('product'),
        pl.col('timestamp').dt.hour().alias('hour')
    ])
    .group_by(['category', 'hour'])
    .agg([
        pl.col('product').mean().alias('avg_product'),
        pl.col('id').count().alias('count')
    ])
    .sort(['category', 'hour'])
)

# 查看優化後的執行計劃
print("查詢計劃:")
print(lazy_query.explain())

# 執行查詢
start = time.time()
result = lazy_query.collect()
print(f"\n懶惰執行耗時: {time.time() - start:.4f} 秒")
print("\n結果預覽:")
print(result.head(10))

## 2. DuckDB - 嵌入式分析型數據庫

### 主要特點
- 🦆 嵌入式 OLAP 數據庫（無需單獨服務器）
- 📊 針對分析查詢優化（類似 SQLite 之於 OLTP）
- 🔍 完整的 SQL 支援
- 💨 列式存儲、向量化執行
- 🔄 直接查詢 Pandas/Polars/Arrow/Parquet

### 何時使用 DuckDB？
- 熟悉 SQL 且偏好使用 SQL 查詢
- 需要複雜的分析查詢（JOIN、窗口函數等）
- 處理 Parquet 文件
- 需要嵌入式分析數據庫

In [ ]:
# DuckDB 基本使用
print("=== DuckDB 基本查詢 ===")

# 創建連接（記憶體數據庫）
con = duckdb.connect(database=':memory:')

# 直接查詢 Pandas DataFrame（無需匯入！）
result = con.execute("""
    SELECT 
        category,
        COUNT(*) as count,
        AVG(value1) as avg_value1,
        AVG(value2) as avg_value2,
        MIN(value1) as min_value1,
        MAX(value1) as max_value1
    FROM df_pandas
    GROUP BY category
    ORDER BY category
""").df()

print(result)
print(f"\n結果類型: {type(result)}")

In [ ]:
# DuckDB 進階查詢：窗口函數
print("=== DuckDB 窗口函數示例 ===")

result = con.execute("""
    SELECT 
        category,
        value1,
        value2,
        ROW_NUMBER() OVER (PARTITION BY category ORDER BY value1 DESC) as rank,
        AVG(value1) OVER (PARTITION BY category) as category_avg,
        value1 - AVG(value1) OVER (PARTITION BY category) as diff_from_avg
    FROM df_pandas
    WHERE value1 > 2.0
    QUALIFY rank <= 5
    ORDER BY category, rank
""").df()

print(result.head(20))

In [ ]:
# 性能比較：DuckDB vs Pandas
print("=== 性能比較：複雜聚合查詢 ===")

# Pandas
start = time.time()
result_pandas = (
    df_pandas[df_pandas['value1'] > 0]
    .groupby('category')
    .agg({
        'id': 'count',
        'value1': ['mean', 'std'],
        'value2': ['sum', 'max']
    })
)
time_pandas = time.time() - start
print(f"Pandas 耗時: {time_pandas:.4f} 秒")

# DuckDB
start = time.time()
result_duckdb = con.execute("""
    SELECT 
        category,
        COUNT(*) as count,
        AVG(value1) as value1_mean,
        STDDEV(value1) as value1_std,
        SUM(value2) as value2_sum,
        MAX(value2) as value2_max
    FROM df_pandas
    WHERE value1 > 0
    GROUP BY category
""").df()
time_duckdb = time.time() - start
print(f"DuckDB 耗時: {time_duckdb:.4f} 秒")

print(f"\n⚡ DuckDB 比 Pandas 快 {time_pandas/time_duckdb:.2f}x")

In [ ]:
# DuckDB 與 Polars 集成
print("=== DuckDB 查詢 Polars DataFrame ===")

# 直接查詢 Polars DataFrame
result = con.execute("""
    SELECT 
        category,
        DATE_TRUNC('hour', timestamp) as hour,
        COUNT(*) as count,
        AVG(value1 * value2) as avg_product
    FROM df_polars
    WHERE value1 > 0
    GROUP BY category, hour
    ORDER BY category, hour
    LIMIT 10
""").df()

print(result)

## 3. 工具選擇指南

### Pandas vs Polars vs DuckDB

| 特性 | Pandas | Polars | DuckDB |
|------|--------|--------|--------|
| **速度** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **記憶體效率** | ⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **易用性** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **SQL 支援** | ❌ | ✅ (有限) | ✅✅ (完整) |
| **生態系統** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **學習曲線** | 低 | 中 | 低-中 |
| **最佳數據量** | <1GB | 1-100GB | 1-1000GB |

### 選擇建議

**使用 Pandas 當：**
- ✅ 數據量小（<1GB）
- ✅ 需要豐富的生態系統
- ✅ 團隊熟悉 Pandas
- ✅ 原型開發或探索性分析

**使用 Polars 當：**
- ✅ 數據量中等（1-100GB）
- ✅ 需要快速數據轉換
- ✅ 偏好 DataFrame API
- ✅ 從 Pandas 遷移（語法相似）
- ✅ 需要充分利用多核 CPU

**使用 DuckDB 當：**
- ✅ 熟悉並偏好 SQL
- ✅ 需要複雜分析查詢
- ✅ 處理 Parquet 文件
- ✅ 需要嵌入式分析數據庫
- ✅ 數據量大（可處理 TB 級）

## 4. 實戰範例：完整數據分析流程

In [ ]:
# 使用 Polars 進行完整的數據分析流程
print("=== Polars 完整數據分析流程 ===")

# 1. 數據載入和清理
df_analysis = (
    df_polars.lazy()
    # 2. 篩選有效數據
    .filter(pl.col('value1').is_not_null())
    # 3. 特徵工程
    .with_columns([
        (pl.col('value1') * pl.col('value2')).alias('product'),
        (pl.col('value1') + pl.col('value2')).alias('sum'),
        pl.col('timestamp').dt.hour().alias('hour'),
        pl.col('timestamp').dt.day_of_week().alias('day_of_week'),
        # 標準化
        ((pl.col('value1') - pl.col('value1').mean()) / pl.col('value1').std()).alias('value1_normalized')
    ])
    # 4. 分組聚合
    .group_by(['category', 'hour'])
    .agg([
        pl.col('id').count().alias('count'),
        pl.col('product').mean().alias('avg_product'),
        pl.col('product').std().alias('std_product'),
        pl.col('value1_normalized').mean().alias('avg_normalized'),
        pl.col('value2').quantile(0.95).alias('value2_95percentile')
    ])
    # 5. 排序
    .sort(['category', 'hour'])
    # 執行
    .collect()
)

print(df_analysis.head(20))
print(f"\n分析結果形狀: {df_analysis.shape}")

In [ ]:
# 使用 DuckDB 進行同樣的分析（用 SQL）
print("=== DuckDB SQL 分析流程 ===")

result_sql = con.execute("""
    WITH enriched_data AS (
        SELECT 
            category,
            value1,
            value2,
            value1 * value2 as product,
            value1 + value2 as sum,
            EXTRACT(HOUR FROM timestamp) as hour,
            EXTRACT(DOW FROM timestamp) as day_of_week,
            (value1 - AVG(value1) OVER ()) / STDDEV(value1) OVER () as value1_normalized
        FROM df_pandas
        WHERE value1 IS NOT NULL
    )
    SELECT 
        category,
        hour,
        COUNT(*) as count,
        AVG(product) as avg_product,
        STDDEV(product) as std_product,
        AVG(value1_normalized) as avg_normalized,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY value2) as value2_95percentile
    FROM enriched_data
    GROUP BY category, hour
    ORDER BY category, hour
""").df()

print(result_sql.head(20))

## 5. 文件 I/O 性能比較

In [ ]:
# 比較不同格式的寫入和讀取性能
import os

print("=== 文件 I/O 性能比較 ===")

# 寫入 Parquet (Pandas)
start = time.time()
df_pandas.to_parquet('/tmp/data_pandas.parquet')
write_time_pandas = time.time() - start
file_size_pandas = os.path.getsize('/tmp/data_pandas.parquet') / 1024**2
print(f"Pandas 寫入 Parquet: {write_time_pandas:.4f} 秒, 文件大小: {file_size_pandas:.2f} MB")

# 寫入 Parquet (Polars)
start = time.time()
df_polars.write_parquet('/tmp/data_polars.parquet')
write_time_polars = time.time() - start
file_size_polars = os.path.getsize('/tmp/data_polars.parquet') / 1024**2
print(f"Polars 寫入 Parquet: {write_time_polars:.4f} 秒, 文件大小: {file_size_polars:.2f} MB")

# 讀取 Parquet (Pandas)
start = time.time()
_ = pd.read_parquet('/tmp/data_pandas.parquet')
read_time_pandas = time.time() - start
print(f"Pandas 讀取 Parquet: {read_time_pandas:.4f} 秒")

# 讀取 Parquet (Polars)
start = time.time()
_ = pl.read_parquet('/tmp/data_polars.parquet')
read_time_polars = time.time() - start
print(f"Polars 讀取 Parquet: {read_time_polars:.4f} 秒")

# 使用 DuckDB 直接查詢 Parquet（無需載入記憶體）
start = time.time()
result = con.execute("""
    SELECT category, COUNT(*), AVG(value1) 
    FROM '/tmp/data_pandas.parquet' 
    GROUP BY category
""").df()
query_time_duckdb = time.time() - start
print(f"DuckDB 直接查詢 Parquet: {query_time_duckdb:.4f} 秒")

print(f"\n⚡ Polars 寫入比 Pandas 快 {write_time_pandas/write_time_polars:.2f}x")
print(f"⚡ Polars 讀取比 Pandas 快 {read_time_pandas/read_time_polars:.2f}x")

## 6. 最佳實踐與建議

### Polars 最佳實踐
1. **使用懶惰執行** - 允許查詢優化
2. **使用表達式 API** - 比鏈式方法更高效
3. **利用 Arrow 格式** - Polars 原生支援
4. **避免行遍歷** - 使用向量化操作
5. **使用 `select` 和 `with_columns`** - 比 `apply` 快得多

### DuckDB 最佳實踐
1. **直接查詢文件** - 無需載入記憶體
2. **使用 CTE** - 提高查詢可讀性
3. **利用列式存儲** - 只讀取需要的列
4. **使用 Parquet** - 最佳的分析型格式
5. **批量操作** - 避免逐行插入

### 遷移建議
- 從 Pandas 遷移到 Polars: 90% 的語法相似
- 混合使用: Polars 可以輕鬆轉換為 Pandas (`to_pandas()`)
- 漸進式採用: 先在性能瓶頸處使用新工具

## 7. 總結

### 關鍵要點
1. **Polars** 是 Pandas 的現代替代品，速度快 10-100 倍
2. **DuckDB** 是嵌入式分析數據庫，適合 SQL 用戶
3. 這些工具可以與 Pandas 混合使用
4. 選擇工具取決於數據量、團隊技能和需求

### 學習資源
- **Polars**: https://pola.rs/
- **DuckDB**: https://duckdb.org/
- **Polars vs Pandas Cheat Sheet**: https://www.rhosignal.com/posts/polars-pandas-cheatsheet/

### 下一步
1. 在小項目中嘗試 Polars 或 DuckDB
2. 比較性能提升
3. 考慮在生產環境中採用
4. 學習更多進階功能（streaming、分區等）